In [1]:
import sys
import os
sys.path.append(os.path.join(os.getcwd(), '..', 'src'))

import pandas as pd
import folium
from folium.plugins import HeatMap
from extractor import load_all_stations
from processor import processar
from stations import carregar_todas_estacoes

print("Bibliotecas carregadas!")

Bibliotecas carregadas!


In [2]:
# Carrega metadados das estações (coordenadas)
df_estacoes = carregar_todas_estacoes()

# Extrai código WMO do nome do arquivo para cruzar com os dados
df_estacoes['CODIGO'] = df_estacoes['ARQUIVO'].str.extract(r'_([A-Z]\d{3})_')

print(df_estacoes[['ESTACAO', 'UF', 'LATITUDE', 'LONGITUDE', 'CODIGO']].head(10))

Lendo metadados de 567 estações...
Estações com coordenadas válidas: 567
              ESTACAO  UF   LATITUDE  LONGITUDE CODIGO
0            BRASILIA  DF -15.789444 -47.925833   A001
1          BRAZLANDIA  DF -15.599722 -48.131111   A042
2     AGUAS EMENDADAS  DF -15.596491 -47.625801   A045
3   GAMA (PONTE ALTA)  DF -15.935278 -48.137500   A046
4  PARANOA (COOPA-DF)  DF -16.012222 -47.557417   A047
5             GOIANIA  GO -16.642778 -49.220278   A002
6           MORRINHOS  GO -17.745066 -49.101698   A003
7           PORANGATU  GO -13.309444 -49.117500   A005
8           SAO SIMAO  GO -18.969142 -50.633449   A011
9            LUZIANIA  GO -16.260556 -47.966944   A012


In [3]:
# Carrega e processa os dados climáticos
extract_dir = os.path.join('..', 'data', '2023')
df_raw = load_all_stations(extract_dir)
df, _ = processar(df_raw)

col_temp = 'TEMPERATURA_DO_AR_-_BULBO_SECO,_HORARIA_C'
df[col_temp] = pd.to_numeric(df[col_temp], errors='coerce')

# Temperatura máxima por arquivo de estação
df_max = df.groupby('ARQUIVO' if 'ARQUIVO' in df.columns else df.index)[col_temp].max().reset_index()
print(df_max.head())
print(df.columns.tolist())

567 arquivos encontrados.
Total de registros carregados: 4,966,920
Iniciando processamento: 4,966,920 registros
Removidos 448,776 registros completamente vazios.
Colunas climáticas identificadas: {'precipitacao': 'PRECIPITACAO_TOTAL,_HORARIO_MM', 'temp_max': 'TEMPERATURA_MAXIMA_NA_HORA_ANT._AUT_C', 'temp_min': 'TEMPERATURA_MINIMA_NA_HORA_ANT._AUT_C'}
Processamento concluído: 4,518,144 registros
   index  TEMPERATURA_DO_AR_-_BULBO_SECO,_HORARIA_C
0      0                                       20.1
1      1                                       19.2
2      2                                       19.3
3      3                                       19.3
4      4                                       19.7
['DATA', 'HORA_UTC', 'PRECIPITACAO', 'PRESSAO_ATMOSFERICA_AO_NIVEL_DA_ESTACAO,_HORARIA_MB', 'PRESSAO_ATMOSFERICA_MAX.NA_HORA_ANT._AUT_MB', 'PRESSAO_ATMOSFERICA_MIN._NA_HORA_ANT._AUT_MB', 'RADIACAO_GLOBAL_KJ_M2', 'TEMPERATURA_DO_AR_-_BULBO_SECO,_HORARIA_C', 'TEMPERATURA_DO_PONTO_DE_ORVALHO_

In [4]:
# Extrai temperatura máxima por estação a partir do nome do arquivo CSV
extract_dir = os.path.join('..', 'data', '2023')
arquivos = [f for f in os.listdir(extract_dir) if f.endswith('.CSV') or f.endswith('.csv')]

from extractor import load_station_file

resultados = []
for arquivo in arquivos:
    filepath = os.path.join(extract_dir, arquivo)
    df_est = load_station_file(filepath)
    if df_est.empty:
        continue
    # Padroniza nome da coluna
    col = [c for c in df_est.columns if 'BULBO' in c.upper() or 'TEM_INS' in c.upper()]
    if not col:
        continue
    temp_max = pd.to_numeric(df_est[col[0]], errors='coerce').max()
    codigo = arquivo.split('_')[3] if len(arquivo.split('_')) > 3 else None
    resultados.append({'CODIGO': codigo, 'TEMP_MAX_EST': temp_max, 'ARQUIVO': arquivo})

df_temp_estacao = pd.DataFrame(resultados)
print(f"Estações processadas: {len(df_temp_estacao)}")
print(df_temp_estacao.head())

Estações processadas: 567
  CODIGO  TEMP_MAX_EST                                            ARQUIVO
0   A001          34.4  INMET_CO_DF_A001_BRASILIA_01-01-2023_A_31-12-2...
1   A042          34.0  INMET_CO_DF_A042_BRAZLANDIA_01-01-2023_A_31-12...
2   A045          36.3  INMET_CO_DF_A045_AGUAS EMENDADAS_01-01-2023_A_...
3   A046          35.7  INMET_CO_DF_A046_GAMA (PONTE ALTA)_01-01-2023_...
4   A047          35.4  INMET_CO_DF_A047_PARANOA (COOPA-DF)_01-01-2023...


In [5]:
# Cruza temperatura máxima com coordenadas das estações
df_mapa = df_estacoes.merge(df_temp_estacao, on='CODIGO', how='inner')
df_mapa = df_mapa.dropna(subset=['LATITUDE', 'LONGITUDE', 'TEMP_MAX_EST'])

print(f"Estações no mapa: {len(df_mapa)}")
print(df_mapa[['ESTACAO', 'UF', 'LATITUDE', 'LONGITUDE', 'TEMP_MAX_EST']].head())

Estações no mapa: 551
              ESTACAO  UF   LATITUDE  LONGITUDE  TEMP_MAX_EST
0            BRASILIA  DF -15.789444 -47.925833          34.4
1          BRAZLANDIA  DF -15.599722 -48.131111          34.0
2     AGUAS EMENDADAS  DF -15.596491 -47.625801          36.3
3   GAMA (PONTE ALTA)  DF -15.935278 -48.137500          35.7
4  PARANOA (COOPA-DF)  DF -16.012222 -47.557417          35.4


In [6]:
# Gera o mapa interativo
import branca.colormap as cm

# Escala de cores: azul (frio) -> amarelo -> vermelho (quente)
colormap = cm.LinearColormap(
    colors=['#2166ac', '#74add1', '#fee090', '#f46d43', '#d73027'],
    vmin=df_mapa['TEMP_MAX_EST'].min(),
    vmax=df_mapa['TEMP_MAX_EST'].max(),
    caption='Temperatura Máxima Registrada (°C) — 2023'
)

# Cria o mapa centrado no Brasil
mapa = folium.Map(
    location=[-15, -50],
    zoom_start=4,
    tiles='CartoDB positron'
)

# Adiciona um ponto por estação
for _, row in df_mapa.iterrows():
    folium.CircleMarker(
        location=[row['LATITUDE'], row['LONGITUDE']],
        radius=7,
        color=colormap(row['TEMP_MAX_EST']),
        fill=True,
        fill_color=colormap(row['TEMP_MAX_EST']),
        fill_opacity=0.85,
        popup=folium.Popup(
            f"<b>{row['ESTACAO']}</b><br>"
            f"UF: {row['UF']}<br>"
            f"Temp. Máxima: {row['TEMP_MAX_EST']:.1f}°C",
            max_width=200
        ),
        tooltip=f"{row['ESTACAO']} — {row['TEMP_MAX_EST']:.1f}°C"
    ).add_to(mapa)

# Adiciona a legenda de cores
colormap.add_to(mapa)

# Salva o mapa
output_path = os.path.join('..', 'data', 'mapa_temperaturas.html')
mapa.save(output_path)
print(f"Mapa salvo em: {output_path}")

Mapa salvo em: ..\data\mapa_temperaturas.html
